# LLM Engineering

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T31SHA/data-ml-handbook/blob/main/notebooks/13_llm_engineering.ipynb)

Runnable examples from **Data/ML Handbook** — section `13_llm_engineering`. Read the explanations in the [docs](https://t31sha.github.io/data-ml-handbook/handbook/13_llm_engineering/).

In [ ]:
# Setup: works in Colab or locally from the repo root
import os, subprocess, sys
if not os.path.exists("code") and not os.path.exists("../code"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/T31SHA/data-ml-handbook.git"], check=True)
    os.chdir("data-ml-handbook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if os.path.exists("../code"): os.chdir("..")
os.chdir("code"); os.makedirs("data", exist_ok=True)
for prep in ["proj01_make_csv.py", "s02_load.py"]:          # shared datasets used by many snippets
    if not os.path.exists("data/housing.parquet"): subprocess.run([sys.executable, prep], check=True)
sys.path.insert(0, "../figures/src")

### `code/s10_llm_math.py`

In [ ]:
def kv_cache_gb(layers, kv_heads, head_dim, seq, batch=1, bytes_=2):
    return 2 * layers * kv_heads * head_dim * seq * batch * bytes_ / 1e9
def weights_gb(params_b, bits): return params_b * 1e9 * bits / 8 / 1e9
print("Llama-3-8B-like (32 layers, 8 KV heads (GQA), head_dim 128):")
for seq in [8_192, 32_768, 128_000]: print(f"  KV-cache @ {seq:>7,} tokens, batch 1, fp16: {kv_cache_gb(32, 8, 128, seq):6.2f} GB")
for bits in [16, 8, 4]: print(f"  weights at {bits:2d}-bit: {weights_gb(8, bits):5.1f} GB")
d = k = 4096; r = 16
print(f"LoRA r={r} on a {d}x{k} matrix: {r*(d+k):,} trainable vs {d*k:,} ({r*(d+k)/(d*k):.2%})")
tokens_in, tokens_out, p_in, p_out = 3_000, 400, 0.15, 0.60        # USD per 1M tokens (example prices)
print(f"cost per request ≈ USD {(tokens_in*p_in + tokens_out*p_out)/1e6:.5f}; per 1M requests ≈ USD {(tokens_in*p_in + tokens_out*p_out):,.0f}")

### `code/s10_quant.py`

In [ ]:
import numpy as np
w = np.random.default_rng(0).normal(0, 0.02, 4096).astype(np.float32)
def quant(w, bits, group=128):
    qmax = 2 ** (bits - 1) - 1; out = np.empty_like(w)
    for g in range(0, len(w), group):                        # per-group symmetric absmax scaling
        blk = w[g:g + group]; s = np.abs(blk).max() / qmax
        out[g:g + group] = np.clip(np.round(blk / s), -qmax - 1, qmax) * s
    return out
for b in [8, 4, 3]:
    e = quant(w, b) - w; print(f"int{b}: relative RMS error = {np.sqrt((e**2).mean())/w.std():.4f}")

### `code/s10_lora.py`

In [ ]:
import torch, torch.nn as nn
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base; self.base.requires_grad_(False)                   # frozen W0
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))            # zero init => starts identical
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)
    def forward(self, x): return self.base(x) + self.drop(x) @ self.A.T @ self.B.T * self.scale
layer = LoRALinear(nn.Linear(1024, 1024))
x = torch.randn(2, 1024)
print("identical at init:", torch.allclose(layer(x), layer.base(x)))
tr = sum(p.numel() for p in layer.parameters() if p.requires_grad); tot = sum(p.numel() for p in layer.parameters())
print(f"trainable {tr:,} of {tot:,} ({tr/tot:.2%})")

### `code/s10_rag.py`

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
docs = {"d1": "Refunds are issued within 14 days of purchase if the item is unused.",
        "d2": "Shipping to Nairobi takes 3-5 business days; express is next day.",
        "d3": "Our API rate limit is 100 requests per minute per key.",
        "d4": "Unused items can be returned for a full refund; opened items get store credit.",
        "d5": "Passwords must be rotated every 90 days for admin accounts."}
ids, texts = list(docs), list(docs.values())
vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(texts); E = vec.transform(texts)
def retrieve(q, k=2):
    s = (E @ vec.transform([q]).T).toarray().ravel(); top = np.argsort(-s)[:k]; return [(ids[i], round(float(s[i]), 3)) for i in top]
gold = {"how long do refunds take": {"d1", "d4"}, "delivery time to nairobi": {"d2"}, "api limits": {"d3"}}
recall, rr = [], []
for q, rel in gold.items():
    got = [d for d, _ in retrieve(q, 2)]; recall.append(len(rel & set(got)) / len(rel))
    rr.append(next((1 / (i + 1) for i, d in enumerate(got) if d in rel), 0)); print(f"{q!r:30s} -> {retrieve(q, 2)}")
print(f"recall@2={np.mean(recall):.2f}  MRR={np.mean(rr):.2f}")
ctx = "\n".join(f"[{d}] {docs[d]}" for d, _ in retrieve("how long do refunds take"))
prompt = f"Answer ONLY from the context and cite [ids]. If missing, say so.\n<context>\n{ctx}\n</context>\nQ: how long do refunds take?"
print(prompt)